# Semantic Kernel 

In this code sample, you will use the [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI Framework to create a basic agent. 

The goal of this sample is to show you the steps that we will later use in the addtional code samples when implementing the different agentic patterns. 

## Import the Needed Python Packages 

In [1]:
import os 
from typing import Annotated
from openai import AsyncOpenAI

from dotenv import load_dotenv

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

## Creating the Client

In this sample, we will use [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) for access to the LLM. 

The `ai_model_id` is defined as `gpt-4o-mini`. Try changing the model to another model available on the GitHub Models marketplace to see the different results. 

For us to use the `Azure Inference SDK` that is used for the `base_url` for GitHub Models, we will use the `OpenAIChatCompletion` connector within Semantic Kernel. There are also other [available connectors](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) to use Semantic Kernel for other model providers.

In [2]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [3]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## Creating the Agent 

Below we create the Agent called `TravelAgent`.

For this example, we are using very simple instructions. You can change these instructions to see how the agent responds differently. 

In [4]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## Running the Agent

Now we can run the Agent by defining a thread of type `ChatHistoryAgentThread`.  Any required system messages are provided to the agent's invoke_stream `messages` keyword argument.

After these are defined, we create a `user_inputs` that will be what the user is sending to the agent. In this case, we have set this message to `Plan me a sunny vacation`. 

Feel free to change this message to see how the agent responds differently. 

In [9]:
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "Plan me 3 day trip in one city. answer korean",
    ]

    for user_input in user_inputs:
        print(f"# User: {user_input}\n")
        first_chunk = True
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            # 5. Print the response
            if first_chunk:
                print(f"# {response.name}: ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread
        print()

    # Clean up the thread
    await thread.delete() if thread else None

await main()

# User: Plan me 3 day trip in one city. answer korean



# TravelAgent: 당신의 3일 여행을 계획할 도시로 "도쿄, 일본"을 추천드립니다! 아래는 도쿄에서의 3일 여행 일정입니다.

### 1일차: 도쿄의 전통 문화
- **오전:** 아사쿠사 방문 - 센소지 사원 탐방 및 나카미세 거리에서 전통 간식 맛보기
- **오후:** 우에노 공원 탐방 - 우에노 동물원이나 박물관 방문
- **저녁:** 스미다 강 유람선 탑승 - 도시 야경 감상하며 저녁 식사

### 2일차: 현대적 도쿄
- **오전:** 신주쿠 교통과 쇼핑 - 신주쿠 교통 센터와 다카시마야 백화점 방문
- **오후:** 하라주쿠와 시부야 탐방 - 메이지 신궁, 다케시타 거리, 시부야 스크램블 교차로 구경
- **저녁:** 시부야 근처의 이자카야에서 저녁 식사

### 3일차: 일본의 자연과 전망
- **오전:** 도쿄 타워 또는 도쿄 스카이트리 방문 - 전경 감상
- **오후:** 신주쿠 교통센터 근처의 신주쿠 교외 공원에서의 산책 및 자연 체험
- **저녁:** 긴자 지역에서 고급 스시 레스토랑에서 저녁 식사

이 일정은 도쿄의 전통과 현대적인 매력을 모두 느낄 수 있도록 구성되었습니다. 즐거운 여행 되세요!


# User: Plan me a day trip.

# TravelAgent: How about a day trip to Bali, Indonesia? Here’s a suggested itinerary for your day in Bali:

### Morning
- **Visit Tegallalang Rice Terraces**: Start your day by exploring the stunning rice terraces. Take a leisurely stroll through the lush green fields and enjoy the beautiful landscape.
- **Breakfast at a Local Café**: Enjoy a traditional Balinese breakfast at one of the local cafés with views of the rice terraces.

### Midday
- **Ubud Monkey Forest**: Head to the Sacred Monkey Forest Sanctuary in Ubud. Walk amidst beautiful temples and ancient trees while observing playful monkeys. 
- **Lunch in Ubud**: Enjoy lunch in Ubud at a local restaurant, trying traditional dishes like Nasi Goreng or Bebek Betutu.

### Afternoon
- **Art Market in Ubud**: After lunch, visit the Ubud Art Market for some shopping. You can find handcrafted souvenirs, clothes, and artwork.
- **Saul’s Spa**: Treat yourself to a massage or spa treatment at one of Ubud’s renowned spas for relaxation.

### Evening
- **Watch the Sunset at Tanah Lot Temple**: End your day by visiting Tanah Lot Temple, which is famous for its stunning ocean views. Arrive in time to watch the sunset behind the temple.
- **Dinner with a View**: Enjoy dinner at a nearby restaurant with views of the ocean where you can savor fresh seafood.

### Travel Tips
- **Transportation**: Consider hiring a private driver or renting a scooter to navigate between the sites easily.
- **Cultural Respect**: Dress modestly when visiting temples; sarongs are often provided at the entrance.

This itinerary will give you a taste of Bali’s natural beauty, culture, and cuisine! Enjoy your trip!